##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini Native Image generation (aka 🍌Nano-Banana models)

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_Started_Nano_Banana.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

---
> **🍌Gemini 3 Models**: If you are only interested in the new [Nano-Banana Pro](https://ai.google.dev/gemini-api/docs/models#gemini-3-pro-image) or [Nano-Banana Pro](https://ai.google.dev/gemini-api/docs/models#gemini-3-pro-image) model, jump directly to the [dedicated section](#gemini-3-models).

---


This notebook will show you how to use the native Image-output feature of Gemini, using the model multimodal capabilities to output both images and texts, and iterate on an image through a discussion.

There are now 3 models you can use:
* `gemini-2.5-flash-image` aka. "nano-banana": Cheap and fast yet powerful. This should be your default choice.
* `gemini-3-pro-image-preview` aka "nano-banana-pro": More powerful thanks to its **thinking** capabilities and its access to real-wold data using **Google Search**. It really shines at creating diagrams and grounded images. And cherry on top, it can create 2K and 4K images!
* `gemini-3.1-flash-image-preview` aka. "nano-banana-2": The best balance between speed and quality, with new capabilities like **Search Grounding**, **Thinking**, and a new **512p** resolution.

These models are really good at:
* **Maintaining character consistency**: Preserve a subject’s appearance across multiple generated images and scenes
* **Performing intelligent editing**: Enable precise, prompt-based edits like inpainting (adding/changing objects), outpainting, and targeted transformations within an image
* **Compose and merge images**: Intelligently combine elements from multiple images into a single, photorealistic composite (maximum 3 with flash, 14 with pro)
* **Leverage multimodal reasoning**: Build features that understand visual context, such as following complex instructions on a hand-drawn diagram

Following this guide, you'll learn how to do all those things and even more.

> **Note:** [Enable billing](https://ai.google.dev/gemini-api/docs/billing#enable-cloud-billing) to use Image Generation. This is a pay-as-you-go feature (cf. [pricing](https://ai.google.dev/pricing#gemini-2.5-flash-image-preview)).


Note that [Imagen](./Get_started_imagen.ipynb) models also offer image generation but in a slightly different way as the Image-out feature has been developed to work iteratively so if you want to make sure certain details are clearly followed, and you are ready to iterate on the image until it's exactly what you envision, Image-out is for you.

Check the [documentation](https://ai.google.dev/gemini-api/docs/image-generation#choose-a-model) for more details on both features and some more advice on when to use each one.

## Setup

In [ ]:
from google.colab import auth
auth.authenticate_user()

### Install SDK

In [ ]:
%pip install -U -q "google-genai>=1.65.0" # minimum version needed for the nano-banana 2 support

### Setup your API key

To run the following cell, your API key must be stored in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for an example.

In [ ]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

### Initialize SDK client

With the new SDK you now only need to initialize a client with your API key (or OAuth if using Vertex AI). The model is now set in each call.

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(
    vertexai=True,
    project='INSERT YOUR PROJECT ID HERE', # Your specific Project ID
    location='us-central1'
)

### Select a model

You can choose between three models:
* `gemini-2.5-flash-image` aka. "nano-banana": Cheap and fast yet powerful. This should be your default choice.
* `gemini-3-pro-image-preview` aka "nano-banana-pro": Has thinking and google search grounding, and can even output 2K and 4K images (cf. [dedicated section](#gemini-3-models))
* `gemini-3.1-flash-image-preview` aka. "nano-banana-2": The latest Flash model with support for Search Grounding, Thinking, and 512p resolution.

In [ ]:
MODEL_ID = "gemini-2.5-flash-image" # @param ["gemini-2.5-flash-image", "gemini-3.1-flash-image-preview", "gemini-3-pro-image-preview"] {"allow-input":true, isTemplate: true}

### Utils

These two functions will help you manage the outputs of the model.

Compared to when you simply generate text, this time the output will contain multiple parts, some one them being text while others will be images. You'll also have to take into account that there could be multiple images so you cannot stop at the first one.


In [ ]:
from IPython.display import display, Markdown
import pathlib

# Loop over all parts and display them either as text or images
def display_response(response):
  for part in response.parts:
    if part.thought: # We don't want to see the thoughts
      continue
    if part.text:
      display(Markdown(part.text))
    elif image:= part.as_image():
      image.show()

  # Display grounding sources if available
  if response.candidates and response.candidates[0].grounding_metadata and response.candidates[0].grounding_metadata.search_entry_point:
      display(HTML(response.candidates[0].grounding_metadata.search_entry_point.rendered_content))

# Save the image
# If there are multiple ones, only the last one will be saved
def save_image(response, path):
  for part in response.parts:
    if image:= part.as_image():
      image.save(path)

##Workshop BONUS Challenge: Kitchen Nightmare Renovation


In [ ]:
import PIL.Image
from google.genai import types

# 1. Download the "Messy Kitchen" photo directly to the session
!wget "https://upload.wikimedia.org/wikipedia/commons/6/68/Messy_kitchen_sink.jpg" -O initial_kitchen.jpg -q

# 2. Start the Chat Mode (ensure MODEL_ID is "gemini-2.5-flash-image")
kitchen_chat = client.chats.create(model=MODEL_ID)

# 3. Define the first renovation step
# Participants can change this prompt to start differently
first_edit_prompt = """Generate a new version of the kitchen that [INSERT YOUR FIRST PROMPT HERE]"""

print("Starting the renovation discussion...")

# 4. Send the message with the reference image to the chat
response = kitchen_chat.send_message(
    message= [
        first_edit_prompt,
        PIL.Image.open('initial_kitchen.jpg') # The anchor image
    ]
)

# 5. Display the result
display_response(response)

# (Optional) Save this base image if you want a fixed backup
save_image(response, "clean_kitchen_base.png")

In [ ]:
# Add a new cell and run this to change materials
message = "Now, in that same clean kitchen, [INSERT YOUR SECOND PROMPT HERE]"
# We do NOT need to pass the image again; the chat remembers it.
response = kitchen_chat.send_message(message)
display_response(response)

In [ ]:
# Add another new cell
message = "In the previous image, [INSERT YOUR LAST PROMPT HERE]"

response = kitchen_chat.send_message(message)
display_response(response)

## Generate images

Using the Gemini Image generation model is the same as using any Gemini model: you simply call `generate_content`.

You can set the `response_modalities` to indicate to the model that you are expecting text and images in the output but it's optional as this is expected with this model.

If you just want an image and don't need text, you can set `response_modalities=['Image']`.

In [ ]:
prompt = 'Create a photorealistic image of a siamese cat with a green left eye and a blue right one and red patches on his face and a black and pink nose' # @param {type:"string"}

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(
        response_modalities=['Text', 'Image'] # response_modalities=['Image'] if you only want the images
    )
)

display_response(response)
save_image(response, 'cat.png')

## Edit images

You can also do image editing, simply pass the original image as part of the prompt. Don't limit yourself to simple edit, Gemini is able to keep the character consistency and reprensent you character in different behaviors or places.

In [ ]:
import PIL

text_prompt = "Create a side view picture of that cat, in a tropical forest, eating a nano-banana, under the stars" # @param {type:"string"}

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        text_prompt,
        PIL.Image.open('cat.png')
    ]
)

display_response(response)
save_image(response, 'cat_tropical.png')

As you can see, you can clearly recognize the same cat with its peculiar nose and eyes.

## Control aspect ratio

You can control the aspect ratio of the output image. The model's primary behavior is to match the size of your input images; otherwise, it defaults to generating square (1:1) images.

To do so, add an `aspect_ratio` value to the `image_config` as you can see in the cell below. The different ratios available and the size of the image generated are listed in this table:

| Aspect ratio | Resolution (1k) |
| --- | --- |
| 1:1 | 1024x1024 |
| 2:3 | 832x1248 |
| 3:2 | 1248x832 |
| 3:4 | 864x1184 |
| 4:3 | 1184x864 |
| 4:5 | 896x1152 |
| 5:4 | 1152x896 |
| 9:16 | 768x1344 |
| 16:9 | 1344x768 |
| 21:9 | 1536x672 |
| 1:4 (NB2) | 121x488 |
| 1:8 (NB2) | 103x864 |
| 4:1 (NB2) | 488x121 |
| 8:1 (NB2) | 864x103 |

Note that the number of tokens stays the same for all aspect ratio and depends only on the model and the resolution.

In [ ]:
import PIL

text_prompt = "Now the cat should keep the same attitude, but be well dressed in fancy restaurant and eat a fancy nano banana." # @param {type:"string"}
aspect_ratio = "16:9" # @param ["1:1","1:4","1:8","2:3","3:2","3:4","4:1","4:3","4:5","5:4","8:1","9:16","16:9","21:9"]

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        text_prompt,
        PIL.Image.open('cat_tropical.png')
    ],
    config=types.GenerateContentConfig(
        response_modalities=["IMAGE"],
        image_config=types.ImageConfig(
            aspect_ratio=aspect_ratio,
        )
    )

)

display_response(response)
save_image(response, 'cat_resaurant.png')

## Get multiple images (ex: tell stories)

So far you've only generated one image per call, but you can request way more than that! Let's try a baking receipe or telling a story.

In [ ]:
prompt = "Show me how to bake macarons with images" # @param ["Show me how to bake macarons with images","Create a beautifully entertaining 8 part story with 8 images with two blue characters and their adventures in the 1960s music scene. The story is thrilling throughout with emotional highs and lows and ending on a great twist and high note. Do not include any words or text on the images but tell the story purely through the imagery itself. "] {"allow-input":true}

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
)

display_response(response)

# Be careful; long output (see below)

The output of the previous code cell could not be saved in the notebook without making it too big to be managed by Github, but here are some examples of what it should look like when you run it when asking for a story, or for a baking receipe:

----------
**Prompt**: *Create a beautifully entertaining 8 part story with 8 images with two blue characters and their adventures in the 1960s music scene. The story is thrilling throughout with emotional highs and lows and ending on a great twist and high note. Do not include any words or text on the images but tell the story purely through the imagery itself.*
![Azure tone story](https://storage.googleapis.com/generativeai-downloads/images/azuretones.png)
(Images have been stitched together)

----------
**Prompt**: *Show me how to bake macarons with images*


That sounds delicious! Here's a simplified guide on how to bake macarons. While it can be a bit tricky, practice makes perfect!

**Ingredients you'll need:**

*   **For the Macaron Shells:**
    *   100g almond flour
    *   100g powdered sugar
    *   75g granulated sugar
    *   2 egg whites (aged for a day or two at room temp, if possible, for better stability)
    *   Pinch of salt (optional)
    *   Food coloring (gel or powder, not liquid)

*   **For the Filling:** (Buttercream, ganache, or jam are popular choices)

---

**Step 1: Prepare your dry ingredients.**
Sift together the almond flour and powdered sugar into a bowl. This step is crucial for achieving smooth macaron shells, as it removes any lumps.


![Macaron prepartation step 1](https://storage.googleapis.com/generativeai-downloads/images/macaron_step1.png)

**Step 2: Make the meringue.**
In a separate, clean bowl, beat the egg whites with a pinch of salt (if using) until foamy. Gradually add the granulated sugar, continuing to beat until you achieve stiff, glossy peaks. If you're using food coloring, add it now. The meringue should be firm enough that you can turn the bowl upside down without it falling out.

![Macaron prepartation step 2](https://storage.googleapis.com/generativeai-downloads/images/macaron_step2.png)

**Step 3: Combine dry ingredients with meringue (Macaronage).**
Gently fold the sifted almond flour and powdered sugar into the meringue in two or three additions. This is called "macaronage" and is the most critical step. You want to mix until the batter flows like "lava" or a slowly ribboning consistency when you lift your spatula. Be careful not to overmix, or your macarons will be flat; under-mixing will result in lumpy shells.

![Macaron prepartation step 3](https://storage.googleapis.com/generativeai-downloads/images/macaron_step3.png)

**Step 4: Pipe the macarons.**
Transfer the batter to a piping bag fitted with a round tip. Pipe uniform circles onto baking sheets lined with parchment paper or silicone mats. Leave some space between each macaron.

![Macaron prepartation step 4](https://storage.googleapis.com/generativeai-downloads/images/macaron_step4.png)

**Step 5: Tap and Rest.**
Firmly tap the baking sheets on your counter several times to release any air bubbles. Use a toothpick to pop any remaining bubbles. This helps create smooth tops and the characteristic "feet." Let the piped macarons rest at room temperature for 30-60 minutes, or until a skin forms on top. When you gently touch a shell, it shouldn't feel sticky. This "drying" step is essential for the feet to develop properly.

![Macaron prepartation step 5](https://storage.googleapis.com/generativeai-downloads/images/macaron_step5.png)

**Step 6: Bake the macarons.**
Preheat your oven to 300°F (150°C). Bake one tray at a time for 12-15 minutes. The exact time can vary by oven. They are done when they have developed "feet" and don't wobble when gently touched.

**Step 7: Cool and Fill.**
Once baked, let the macaron shells cool completely on the baking sheet before carefully peeling them off. This prevents them from breaking.  Then, match them up by size and pipe or spread your chosen filling onto one shell before sandwiching it with another.

![Macaron prepartation step 7](https://storage.googleapis.com/generativeai-downloads/images/macaron_step7.png)

Finally, let them mature in the refrigerator for at least 24 hours. This allows the flavors to meld and the shells to soften to the perfect chewy consistency.

Enjoy your homemade macarons!

-----

## Chat mode (recommended method)

So far you've used unary calls, but Image-out is actually made to work better with chat mode as it's easier to iterate on an image turn after turn.

In [ ]:
chat = client.chats.create(
    model=MODEL_ID,
)

In [ ]:
message = "create a image of a plastic toy fox figurine in a kid's bedroom, it can have accessories but no weapon" # @param {type:"string"}

response = chat.send_message(message)
display_response(response)
save_image(response, "figurine.png")

In [ ]:
message = "Add a blue planet on the figuring's helmet or hat (add one if needed)" # @param {type:"string"}
response = chat.send_message(message)
display_response(response)

In [ ]:
message = 'Move that figurine on a beach' # @param {type:"string"}
response = chat.send_message(message)
display_response(response)

In [ ]:
message = 'Now it should be base-jumping from a spaceship with a wingsuit' # @param {type:"string"}
response = chat.send_message(message)
display_response(response)

In [ ]:
message = 'Cooking a barbecue with an apron' # @param {type:"string"}
response = chat.send_message(message)
display_response(response)

In [ ]:
message = 'What about chilling in a spa?' # @param {type:"string"}
response = chat.send_message(message)
display_response(response)

You can also control the aspect ratio of the output image in chat mode.

To do so, add an `aspect_ratio` value to the `image_config` as you can see in the cell below.

In [ ]:
message = "Bring it back to the bedroom" # @param {type:"string"}
response = chat.send_message(
    message,
    config=types.GenerateContentConfig(
        image_config=types.ImageConfig(aspect_ratio="16:9"),
    ),
)
display_response(response)

## Mix multiple pictures

You can also mix multiple images (up to 3 with nano-banana, 14 with nano-banana-pro, 6 with high fidelity), either because there are multiple characters in your image, or because you want to hightlight a certain product, or set the background.

In [ ]:
import PIL

text_prompt = "Create a picture of that figurine riding that cat in a fantasy world." # @param {type:"string"}

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        text_prompt,
        PIL.Image.open('cat.png'),
        PIL.Image.open('figurine.png')
    ],
)

display_response(response)

<a name="nano-banana-pro"></a>
## Gemini 3 Models (Nano-Banana Pro & 2)

Reflecting their shared origin from Gemini 3 models, both **Nano-Banana Pro** and **Nano-Banana 2** offer advanced capabilities beyond the standard Flash model.

They both support [**thinking**](#thinking), allowing them to process complex requests more effectively. Additionally, they can use [**Search Grounding**](#grounding) to access up-to-date information and provide more accurate responses.

**Nano-Banana Pro** really shines at creating diagrams and can even output **2K and 4K images** (cf. [dedicated section](#image_size)).

**Nano-Banana 2** provides a great balance between speed and quality, introduces a low-latency **512p resolution** mode and is able to even better ground its requests using google search.

In [ ]:
# @title Run this cell to set everything up (especially if you jumped directly to this section)

from google.colab import userdata
from google import genai
from google.genai import types
from IPython.display import display, Markdown, HTML
import PIL

client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

# Loop over all parts and display them either as text or images
def display_response(response):
  for part in response.parts:
    if part.thought: # We don't want to see the thoughts
      continue
    if part.text:
      display(Markdown(part.text))
    elif image:= part.as_image():
      image.show()

# Save the image
# If there are multiple ones, only the last one will be saved
def save_image(response, path):
  for part in response.parts:
    if image:= part.as_image():
      image.save(path)

In [ ]:
# Let's switch to the 3.1 model
GEMINI3_MODEL_ID = "gemini-3.1-flash-image-preview" # @param ["gemini-3.1-flash-image-preview", "gemini-3-pro-image-preview"] {"allow-input":true, isTemplate: true}

<a name="thinking" ></a>

### Check the thoughts (Nano-Banana Pro et Nano-Banana 2)

Gemini 3 models can "think" before they answer. This is particularly useful for complex reasoning tasks.

**Nano-Banana 2** also introduces **Thinking Levels**, allowing you to control the depth of reasoning (available via the `thinking_config` in your request) like the Gemini 3 code models.

Note that you are paying for the thought token (but not the images in them) as output tokens (cf. pricing).

In [ ]:
prompt = "Create an unusual but realistic image that might go viral"  # @param {type:"string"}
aspect_ratio = "16:9" # @param ["1:1","1:4","1:8","2:3","3:2","3:4","4:1","4:3","4:5","5:4","8:1","9:16","16:9","21:9"]
thinking_level = "High" # @param ["Minimal", "High"] # Only for Nano-Banana 2

response = client.models.generate_content(
    model="gemini-3-pro-image-preview", # Changed GEMINI3_MODEL_ID to MODEL_ID to use gemini-2.5-flash-image
    contents=prompt,
    config=types.GenerateContentConfig(
        response_modalities=['Text', 'Image'],
        image_config=types.ImageConfig(
            aspect_ratio=aspect_ratio,
        ),
        thinking_config=types.ThinkingConfig(
            thinking_level=thinking_level,
            include_thoughts=True # Don't forget this part if you want to check the thoughts later
        )
    )
)

display_response(response)
save_image(response, 'viral.png')

Since Nano-banana Pro is a thinking model, you can check the thoughts that led to the image being produced.

In [ ]:
for part in response.parts:
  if part.thought:
    if part.text:
      display(Markdown(part.text))
    elif image:= part.as_image():
      #image.show() # Skipping it since in most case it should be the same image as in the output
      print("IMAGE")

#### Thoughts signatures

The output part of Gemini 3 models always contain `though_signatures`.

If you are using the SDK since it's entirerly managed by the SDKs. But if you are curious, here's what happening behind the scenes.

In [ ]:
# Loop over all parts and display the thought signature:
for part in response.parts:
  if part.thought_signature:
    print(part.thought_signature)

This signature is used by the model when you want to do chat/multi-turn discussions. It helps the model not only remember what was said before, but also what it thought before or what it got from its tools and function calls.

Here's a example: imagine you ask the model for the temperature today (like in the next example). It will do use google search to get the weather then it will tell you that it will be 25°C. If you then ask to add the humidity to the image, thanks to the thought signatures, it will be able to remember that it also got that info from the first call and not do a new request.

More details in the [documentation](https://ai.google.dev/gemini-api/docs/image-generation#thought-signatures).

<a name="grounding"></a>

### Use search grounding (Nano-Banana Pro et Nano-Banana 2)

Note that it only ground using the text results and not the images that could be found using Google Search. You just need to add `tools=[{"google_search": {}}]` to your config.

In [ ]:
prompt = "Visualize the current weather forecast for the next 5 days in Tokyo as a clean, modern weather chart. add a visual on what i should wear each day"  # @param {type:"string"}
aspect_ratio = "16:9" # @param ["1:1","1:4","1:8","2:3","3:2","3:4","4:1","4:3","4:5","5:4","8:1","9:16","16:9","21:9"]

response = client.models.generate_content(
    model=GEMINI3_MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(
        response_modalities=['Text', 'Image'], # Image only currently doesn't wortk with grounding
        image_config=types.ImageConfig(
            aspect_ratio=aspect_ratio,
        ),
        tools=[{"google_search": {}}]
    )
)

display_response(response)
save_image(response, 'weather.png')

Don't forget to display the sources:

In [ ]:
from IPython.display import HTML

display(HTML(response.candidates[0].grounding_metadata.search_entry_point.rendered_content))

More on how Search grounding works in the [dedicated guide ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](./Search_Grounding.ipynb).

<a name="image_grounding"></a>
### Use Image grounding (Nano-Banana 2)

Nano-Banana 2 is also able to seach for images on Google Search to even ebtter ground its generations. This is especially useful when looking for real-world place or landmarks, specific animal species...

If you only provide the `google_search` tool without any parameters like in the previous example, it will only look for text grounding. If you also want it to look for images, you need to specify its `search_types` and tell it to look for images by adding `image_search=types.ImageSearch()`. You can tell to serach for both as in the example below by also adding `web_search=types.WebSearch()`. The pricing is the same wether you ask for one or both.

Note that you can't look for people images.

In [ ]:
PROMPT = "A detailed painting of a Timareta Thelxione butterfly resting on a flower" # @param {type:"string"}

response = client.models.generate_content(
    model=GEMINI3_MODEL_ID,
    contents=PROMPT,
    config=types.GenerateContentConfig(
        response_modalities=["IMAGE"],
        tools=[
            types.Tool(google_search=types.GoogleSearch(
                search_types=types.SearchTypes(
                    web_search=types.WebSearch(),
                    image_search=types.ImageSearch() # This adds the image search grounding (exclusive to the Nano-Banana 2 model)
                )
            ))
        ]
    )
)

display_response(response)

<a name="image_size"></a>
### Generate 4K images (Nano-Banana Pro et Nano-Banana 2)

The gemini 3 models can generate 1K, 2K or 4K images. Nano-Banana 2 can also create 512px ones.

4K images are more expensive so only do it when needed (cf. [pricing](https://ai.google.dev/gemini-api/docs/pricing#gemini-3.1-flash-image-preview)).

Here are the corresponding resolutions for each aspect ratio and resolution:
| Aspect ratio | 512px resolution | 1K resolution | 2K resolution | 4K resolution |
| :--- | :--- | :--- | :--- | :--- |
| **1:1** | 512x512 | 1024x1024 | 2048x2048 | 4096x4096 |
| **2:3** | 424x632 | 848x1264 | 1696x2528 | 3392x5056 |
| **3:2** | 632x424 | 1264x848 | 2528x1696 | 5056x3392 |
| **3:4** | 448x600 | 896x1200 | 1792x2400 | 3584x4800 |
| **4:3** | 600x448 | 1200x896 | 2400x1792 | 4800x3584 |
| **4:5** | 464x576 | 928x1152 | 1856x2304 | 3712x4608 |
| **5:4** | 576x464 | 1152x928 | 2304x1856 | 4608x3712 |
| **9:16** | 384x688 | 768x1376 | 1536x2752 | 3072x5504 |
| **16:9** | 688x384 | 1376x768 | 2752x1536 | 5504x3072 |
| **21:9** | 792x336 | 1584x672 | 3168x1344 | 6336x2688 |
| **1:4** (NB2) | 256x1024 | 512x2048 | 1024x4096 | 2048x8192 |
| **4:1** (NB2) | 1024x256 | 2048x512 | 4096x1024 | 8192x2048 |
| **1:8** (NB2) | 192x1536 | 384x3072 | 768x6144 | 1536x12288 |
| **8:1** (NB2) | 1536x192 | 3072x384 | 6144x768 | 12288x1536 |

Tokens are independent of the aspect ratio and only depends on the model and the resolution. Check the [pricing](https://ai.google.dev/gemini-api/docs/pricing#gemini-3.1-flash-image-preview) for more details.

In [ ]:
prompt = "A photo of an oak tree experiencing every season"  # @param {type:"string"}
aspect_ratio = "1:1" # @param ["1:1","1:4","1:8","2:3","3:2","3:4","4:1","4:3","4:5","5:4","8:1","9:16","16:9","21:9"]
resolution = "4K" # @param ["1K", "2K", "4K"]

response = client.models.generate_content(
    model=GEMINI3_MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(
        response_modalities=['Text', 'Image'],
        image_config=types.ImageConfig(
            aspect_ratio=aspect_ratio,
            image_size=resolution
        )
    )
)

display_response(response)

<a name="resolution_512p"></a>
### Low-latency 512p resolution (Nano-Banana 2)

**Nano-Banana 2** introduces a new **512p** resolution mode, optimized for speed and low-latency applications while maintaining high quality for many use cases.

Tip: use 512px resolution in conjuction with the Batch API to lower your costs to the minimum when you need to create a lot of images and they ask Nano-Banana to upscale the ones you need at higher resolutions.

In [ ]:
RESOLUTION = "512px" # @param ["512px", "1K", "2K", "4K"]

response = client.models.generate_content(
    model="gemini-3.1-flash-image-preview",
    contents="A cute pixel art robot",
    config=types.GenerateContentConfig(
        response_modalities=["IMAGE"],
        image_config=types.ImageConfig(
            image_size=RESOLUTION
        )
    )
)

display_response(response)

<a name="translate"></a>
### Generate or translate image (Nano-Banana Pro & Nano-Banana 2)

You can now generate or translate images is over a dozen languages!

In [ ]:
chat = client.chats.create(
    model=GEMINI3_MODEL_ID,
    config=types.GenerateContentConfig(
        response_modalities=['Text', 'Image'],
        tools=[{"google_search": {}}]
    )
)

In [ ]:
message = "Make an infographic explaining Einstein's theory of General Relativity suitable for a 6th grader in Spanish" # @param {type:"string"}
aspect_ratio = "16:9" # @param ["1:1","1:4","1:8","2:3","3:2","3:4","4:1","4:3","4:5","5:4","8:1","9:16","16:9","21:9"]

response = chat.send_message(message,
    config=types.GenerateContentConfig(
        image_config=types.ImageConfig(
            aspect_ratio=aspect_ratio,
        )
    )
)

display_response(response)
save_image(response, "relativity_ES.png")

In [ ]:
message = "Translate this infographic in Japanese, keeping everything else the same" # @param {type:"string"}
resolution = "2K" # @param ["1K", "2K", "4K"]

response = chat.send_message(message,
    config=types.GenerateContentConfig(
        image_config=types.ImageConfig(
            image_size=resolution
        )
    )
)
display_response(response)
save_image(response, "relativity_JP.png")

<a name="mix" ></a>

### Mix up to 14 images! (Nano-Banana Pro & Nano-Banana 2)

You can now mix up to 6 images in high-fidelity and 14 with minor changes. Here's an example:

In [ ]:
# Get some images
!wget "https://storage.googleapis.com/generativeai-downloads/images/sweets.png" -O "sweets.png" -q
!wget "https://storage.googleapis.com/generativeai-downloads/images/car.png" -O "car.png" -q
!wget "https://storage.googleapis.com/generativeai-downloads/images/rabbit.png" -O "rabbit.png" -q
!wget "https://storage.googleapis.com/generativeai-downloads/images/spartan.png" -O "spartan.png" -q
!wget "https://storage.googleapis.com/generativeai-downloads/images/cactus.png" -O "cactus.png" -q
!wget "https://storage.googleapis.com/generativeai-downloads/images/cards.png" -O "cards.png" -q


In [ ]:
prompt = "Create a marketing photoshoot of those items from my daughter's bedroom. Focus on the items and ignore their backgrounds." # @param {type:"string"}
aspect_ratio = "5:4" # @param ["1:1","1:4","1:8","2:3","3:2","3:4","4:1","4:3","4:5","5:4","8:1","9:16","16:9","21:9"]
resolution = "1K" # @param ["1K", "2K", "4K"]

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        prompt,
        PIL.Image.open('sweets.png'),
        PIL.Image.open('car.png'),
        PIL.Image.open('rabbit.png'),
        PIL.Image.open('spartan.png'),
        PIL.Image.open('cactus.png'),
        PIL.Image.open('cards.png'),
    ],
    config=types.GenerateContentConfig(
        response_modalities=['Text', 'Image'],
        image_config=types.ImageConfig(
            aspect_ratio=aspect_ratio,
            image_size=resolution
        ),
    )
)

display_response(response)

## Other cool prompts to test

### Back to the 80s

In [ ]:
text_prompt = "Create a photograph of the person in this image as if they were living in the 1980s. The photograph should capture the distinct fashion, hairstyles, and overall atmosphere of that time period." # @param {type:"string"}

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        text_prompt,
        PIL.Image.open('cat.png')
    ]
)

display_response(response)
save_image(response, 'cat_80s.png')

### Mini-figurine

In [ ]:
text_prompt = "create a 1/7 scale commercialized figurine of the characters in the picture, in a realistic style, in a real environment. The figurine is placed on a computer desk. The figurine has a round transparent acrylic base, with no text on the base. The content on the computer screen is a 3D modeling process of this figurine. Next to the computer screen is a toy packaging box, designed in a style reminiscent of high-quality collectible figures, printed with original artwork. The packaging features two-dimensional flat illustrations." # @param {type:"string"}

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        text_prompt,
        PIL.Image.open('cat_80s.png')
    ]
)

display_response(response)

### Stickers

In [ ]:
text_prompt = "Create a single sticker in the distinct Pop Art style. The image should feature bold, thick black outlines around all figures, objects, and text. Utilize a limited, flat color palette consisting of vibrant primary and secondary colors, applied in unshaded blocks, but maintain the person skin tone. Incorporate visible Ben-Day dots or halftone patterns to create shading, texture, and depth. The subject should display a dramatic expression. Include stylized text within speech bubbles or dynamic graphic shapes to represent sound effects (onomatopoeia). The overall aesthetic should be clean, graphic, and evoke a mass-produced, commercial art sensibility with a polished finish. The user's face from the uploaded photo must be the main character, ideally with an interesting outline shape that is not square or circular but closer to a dye-cut pattern" # @param {type:"string"}

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        text_prompt,
        PIL.Image.open('cat_80s.png')
    ]
)

display_response(response)

### Multi-image fusion
Tip: Combine multiple images into a single ‘collage’ first if you need to go beyond the image upload limit.


In [ ]:
text_prompt = "Combine everything in these images to create a 60s inspired fashion editorial photoshoot" # @param {type:"string"}

!wget "https://storage.googleapis.com/generativeai-downloads/images/Multiple_images.png" -O "Multiple_images.png" -q
display(Image('Multiple_images.png'))

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        text_prompt,
        PIL.Image.open('cat.png'),
        PIL.Image.open('Multiple_images.png')
    ]
)

display_response(response)

### Colorize black and white images

In [ ]:
text_prompt = "Restore and colorize this image from 1932." # @param {type:"string"}

# Thanks to the wikimedia foundation for hosting this historical picture
!wget "https://upload.wikimedia.org/wikipedia/commons/thumb/9/9c/Lunch_atop_a_Skyscraper_-_Charles_Clyde_Ebbets.jpg/1374px-Lunch_atop_a_Skyscraper_-_Charles_Clyde_Ebbets.jpg" -O "Lunch_atop_a_Skyscraper.jpg" -q
display(Image('Lunch_atop_a_Skyscraper.jpg'))

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        text_prompt,
        PIL.Image.open('Lunch_atop_a_Skyscraper.jpg')
    ]
)

display_response(response)

### Google Map transformation


In [ ]:
text_prompt = "Show me what we see from the red arrow" # @param {type:"string"}

!wget "https://storage.googleapis.com/generativeai-downloads/images/Mont_St_Michel.png" -O "Mont_St_Michel.png" -q
display(Image('Mont_St_Michel.png'))

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        text_prompt,
        PIL.Image.open('Mont_St_Michel.png')
    ]
)

display_response(response)

### Isometric landmark


In [ ]:
text_prompt = "Take this location and make the landmark an isometric image (building only), in the style of the game Theme Park." # @param {type:"string"}

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        text_prompt,
        PIL.Image.open('Mont_St_Michel.png')
    ]
)

display_response(response)

### What does Google know about me? (Nano-Banana Pro & Nano-Banana 2)


In [ ]:
text_prompt = "Search the web then generate an image of isometric perspective, detailed pixel art that shows the career of Guillaume Vernade" # @param {type:"string"}

response = client.models.generate_content(
    model=GEMINI3_MODEL_ID,
    contents=[
        text_prompt,
    ],
    config=types.GenerateContentConfig(
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
        ),
        tools=[{"google_search": {}}]
    )
)

display_response(response)

In [ ]:
display(HTML(response.candidates[0].grounding_metadata.search_entry_point.rendered_content))

### Text-heavy images (Nano-Banana Pro & Nano-Banana 2)


In [ ]:
text_prompt = "Show me an infographic about how sonnets work, using a sonnet about bananas written in it, along with a lengthy literary analysis of the poem. good vintage aesthetics" # @param {type:"string"}

response = client.models.generate_content(
    model=GEMINI3_MODEL_ID,
    contents=[
        text_prompt,
    ],
    config=types.GenerateContentConfig(
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
        ),
    )
)

display_response(response)

### Theater program (Nano-Banana Pro & Nano-Banana 2)


In [ ]:
text_prompt = "A photo of a program for the Broadway show about TCG players on a nice theater seat, it's professional and well made, glossy, we can see the cover and a page showing a photo of the stage." # @param {type:"string"}

response = client.models.generate_content(
    model=GEMINI3_MODEL_ID,
    contents=[
        text_prompt,
    ],
    config=types.GenerateContentConfig(
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
        ),
    )
)

display_response(response)

### Famous meme restyling (Nano-Banana Pro or Nano-Banana 2 & chat)


In [ ]:
text_prompt = "There's a vary famous meme about a dog in a house in fire saying \"this is fine\", can you do a papier maché version of it?" # @param {type:"string"}

chat = client.chats.create(
    model=GEMINI3_MODEL_ID,
    config=types.GenerateContentConfig(
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
        ),
        tools=[{"google_search": {}}]
    )
)

response = chat.send_message(text_prompt)
display_response(response)

other_style = "Now do a new version with generic building blocks" # @param {type:"string"}

response = chat.send_message(other_style)
display_response(response)

other_style = "What about a crochet version?" # @param {type:"string"}

response = chat.send_message(other_style)
display_response(response)

### Sprites (Nano-Banana Pro & Nano-Banana 2)

In [ ]:
import PIL

text_prompt = "Sprite sheet of a jumping illustration, 3x3 grid, white background, sequence, frame by frame animation, square aspect ratio. Follow the structure of the attached reference image exactly." # @param ["Sprite sheet of a jumping illustration, 3x3 grid, white background, sequence, frame by frame animation, square aspect ratio. Follow the structure of the attached reference image exactly.","Sprite sheet of a woman dancing on a drone, 3x3 grid, sequence, frame by frame animation, square aspect ratio. Follow the structure of the attached reference image exactly.","Sprite sheet of oh no, 3x3 grid, white background, sequence, frame by frame animation, square aspect ratio. Follow the structure of the attached reference image exactly."] {"allow-input":true}

!wget "https://storage.googleapis.com/generativeai-downloads/images/grid_3x3_1024.png" -O "grid_3x3_1024.png" -q

response = client.models.generate_content(
    model=GEMINI3_MODEL_ID,
    contents=[
        text_prompt,
        PIL.Image.open("grid_3x3_1024.png")
    ],
    config=types.GenerateContentConfig(
        image_config=types.ImageConfig(
            aspect_ratio="1:1",
        ),
    )
)

display_response(response)
save_image(response, 'sprites.png')

Now let's convert it into a GIF.

In [ ]:
# @title
import PIL
from IPython.display import display, Image

image = PIL.Image.open('sprites.png')

total_width, total_height = image.size

effective_width = total_width - 2
effective_height = total_height - 2

frame_width = effective_width // 3
frame_height = effective_height // 3

frames = []

for row in range(3):
    for col in range(3):
        left = col * (frame_width + 1)
        upper = row * (frame_height + 1)
        right = left + frame_width
        lower = upper + frame_height

        cropped_frame = image.crop((left, upper, right, lower))

        frames.append(cropped_frame)

frames[0].save(
    'sprite.gif',
    save_all=True,
    append_images=frames[1:],
    duration=200,
    loop=0
)

# Display the GIF by reading its bytes, as suggested
display(Image(data=open('sprite.gif','rb').read()))

## Next Steps
### Useful documentation references:

Check the [documentation](https://ai.google.dev/gemini-api/docs/image-generation#gemini) for more details about the image generation capabilities of the model. To improve your prompting skills, check out the [prompt guide](https://ai.google.dev/gemini-api/docs/image-generation#prompt-guide) for great advices on creating your prompts.

### Check a more complex example
[Illustrate a book ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../examples/Book_illustration.ipynb): Use Gemini to create illustration for an open-source book and audiobook

### Play with the AI Studio apps

AI Studio features a ton of Nano-banana Apps that you can test and customize to your needs. Here are my favorite:
* [Past Forward](https://aistudio.google.com/apps/bundled/past_forward) lets you travel through time
* [Personalized comics](https://aistudio.google.com/apps/bundled/personalized_comics) lets you create comics where YOU are the hero (or the foe, or both 🤯)
* [Pixshop](https://aistudio.google.com/apps/bundled/pixshop), an AI-powered image editor
* [FitCheck](https://aistudio.google.com/apps/bundled/fitcheck), let you virtually try on any clothes
* [Info Genius](https://aistudio.google.com/apps/bundled/info_genius), to create infographics of anything
* And [plenty others](https://aistudio.google.com/apps?source=showcase&showcaseTag=nano-banana)

### Check-out Imagen as well:
The [Imagen](https://ai.google.dev/gemini-api/docs/imagen) model is another way to generate images. Check out the [Get Started with Imagen notebook ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](./Get_started_imagen.ipynb) to start playing with it too.

### Continue your discovery of the Gemini API

Gemini is not only good at generating images, but also at understanding them. Check the [Spatial understanding ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](./Spatial_understanding.ipynb) guide for an introduction on those capabilities, and the [Video understanding ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](./Video_understanding.ipynb) one for video examples.
